In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency
from statsmodels.stats.weightstats import ztest
import sys
import os

sys.path.append("..")

In [2]:
df = pd.read_csv("../data/insurance_data_clean.csv")

df.head()

C:\Users\YOGA 9I\AppData\Local\Temp\ipykernel_26852\1881377468.py:1: DtypeWarning: Columns (0: CapitalOutstanding) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/insurance_data_clean.csv")


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [3]:
df["ClaimOccurred"] = (df["TotalClaims"] > 0).astype(int)

df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

df["ClaimSeverity"] = np.where(
    df["TotalClaims"] > 0,
    df["TotalClaims"],
    np.nan
)

In [4]:
province_df = df[df["Province"].isin(["Gauteng", "Northern Cape"])]

table = pd.crosstab(
    province_df["Province"],
    province_df["ClaimOccurred"]
)

table

ClaimOccurred,0,1
Province,,
Gauteng,392543,1322
Northern Cape,6372,8


In [6]:
from src.hypothesis_tests import run_chi_square, decision

stat, p = run_chi_square(table)

print("p-value:", p)
print(decision(p))

p-value: 0.00534907280471951
Reject H₀


In [7]:
df["PostalCode"].value_counts().head(10)

PostalCode
2000    133498
122      49171
7784     28585
299      25546
7405     18518
458      13775
8000     11794
2196     11048
470      10226
7100     10161
Name: count, dtype: int64

In [8]:
zip_df = df[df["PostalCode"].isin([2000, 8001])]

In [9]:
table = pd.crosstab(zip_df["PostalCode"], zip_df["ClaimOccurred"])

stat, p = run_chi_square(table)

print(p)
print(decision(p))

0.6592711885830258
Fail to Reject H₀


In [11]:
from src.hypothesis_tests import run_ttest, run_ztest, run_chi_square, decision

In [12]:
zip1 = df[df["PostalCode"] == 2000]["Margin"]
zip2 = df[df["PostalCode"] == 8001]["Margin"]

stat, p = run_ttest(zip1, zip2)

print(p)
print(decision(p))

0.9670204139985747
Fail to Reject H₀
